# FarmPulse - Model comparison

Notebook مستقل عن `03_transfer_learning.ipynb` ويعرض مقارنة CNN baseline وEfficientNet-B0 وDenseNet121 من MLflow. لا يعيد التدريب. شغّل التدريب والتقييم commands أدناه في Terminal أولاً.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import mlflow
import pandas as pd
from src.utils.tracking import mlflow_tracking_uri

mlflow.set_tracking_uri(mlflow_tracking_uri(ROOT))
client = mlflow.MlflowClient()

## Training and final evaluation commands

Run these from the repository root. EfficientNet and DenseNet use identical data splits and training hyperparameters by default.

```powershell
python -m src.models.baseline --epochs 10
python -m src.models.train --architecture efficientnet_b0 --epochs 10
python -m src.models.train --architecture densenet121 --epochs 10
python -m src.models.evaluate --checkpoint models/best.pt --split test
python -m src.models.evaluate --checkpoint models/densenet121_best.pt --split test
python -m src.models.evaluate --checkpoint models/baseline.pt --split test
```

Use the validation metrics to choose hyperparameters; use test metrics for the final model comparison.

In [ ]:
experiments = client.search_experiments()
runs = []
for experiment in experiments:
    runs.extend(client.search_runs([experiment.experiment_id], max_results=200))

rows = []
for run in runs:
    params = run.data.params
    architecture = params.get('architecture')
    if architecture not in {'baseline_cnn', 'efficientnet_b0', 'densenet121'}:
        continue
    split = params.get('split', 'validation')
    rows.append({
        'architecture': architecture,
        'evaluation_split': split,
        'run_name': run.info.run_name,
        'status': run.info.status,
        'accuracy': run.data.metrics.get('accuracy'),
        'weighted_f1': run.data.metrics.get('f1_weighted'),
        'training_time_seconds': run.data.metrics.get('training_time_seconds'),
        'dataset_version': params.get('dataset_version', 'not logged by this older run'),
    })

comparison = pd.DataFrame(rows)
comparison.sort_values(['evaluation_split', 'weighted_f1'], ascending=[True, False]) if not comparison.empty else comparison

Each new training run records architecture, hyperparameters, dataset fingerprint, per-epoch train loss/validation metrics, total training time, checkpoint, and metrics JSON as MLflow artifacts. It also registers a version with the `staging` alias. To promote a chosen model version after reviewing test results, use `python -m src.models.promote --architecture densenet121 --version 1 --stage Production`. DVC stages are defined for preparation, the three training options, evaluation, and ONNX export.